# Predicting what you were not shown — I-JEPA, and a loss that lies

Notebook 01 built one block with switches and got SimCLR, MoCo, BYOL and SimSiam out of it.
Notebook 02 measured something that block cannot explain: a frozen **MoCo** encoder localised
25 points *worse* than a generic supervised one, on its own domain. The rule that fell out was

> the level your objective operates at is the level your features are good at.

MoCo's objective is image-level: make two crops of a tile agree. To do that the network must
discard *where things are*, because that is exactly what differs between two crops.

This notebook builds the alternative — an objective that works **per patch** — and tests that
prediction. It also runs into the failure mode notebook 01 introduced, hard enough that we need
a hyperparameter search to get out of it.

## Where notebook 01's abstraction ends

`vectormesh.components.Siamese` covers the augmentation-based family completely. It does not
cover this one, and the reason is worth seeing:

| | predictor signature |
|---|---|
| BYOL / SimSiam | `vector -> vector` — "guess what the other view looks like" |
| **I-JEPA** | `context + position -> representation at that position` — "guess what patch 37 looks like, given patches 0-20" |

`Siamese(online, momentum, predictor, stopgrad)` has nowhere to put *which position*. So I-JEPA
is a sibling of that block, not a setting on it — and being able to say precisely where an
abstraction stops is worth more than an abstraction that pretends to cover everything.

In [ ]:
import os

# ray 2.55 detects a `uv run` parent and tries to rebuild its environment on every
# worker, which needs pyproject.toml inside the working directory -- never true for a
# notebook in a subfolder. One line, before ray is imported anywhere.
os.environ["RAY_ENABLE_UV_RUN_RUNTIME_ENV"] = "0"

import copy
import random
import time

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch import Tensor, nn
from torchvision.transforms import v2

from notebooktester import param

torch.manual_seed(0)

IMG, PATCH = 64, 8            # EuroSAT tiles, 8x8 grid of 8-pixel patches
GRID = IMG // PATCH
DIM, DEPTH, HEADS = 192, 4, 3
BATCH = 128

# `param(full, test=...)` returns the small value when notebooktester runs this
# notebook, and the real one when you do. Everything that costs time uses it.
STEPS = param(1500, test=60)          # steps per training run
SEARCH_STEPS = param(400, test=40)    # steps per search trial (shorter on purpose)
N_TRIALS = param(16, test=2)          # ray trials per search
FINAL_STEPS = param(3000, test=60)    # the winning config, trained properly
N_TRAIN = param(6000, test=600)
PROBE_N = param(1200, test=300)   # labelled tiles used to score a search trial

print(f"STEPS={STEPS}  SEARCH_STEPS={SEARCH_STEPS}  N_TRIALS={N_TRIALS}  FINAL_STEPS={FINAL_STEPS}")

## 1. Views by masking, not by augmenting

Every method in notebook 01 made its two views with *augmentation* — crop it twice, and declare
the results the same thing. I-JEPA makes them by **hiding part of the image**:

- the **context** is the patches you are allowed to see
- the **targets** are a few blocks you are not

and the task is to produce, for each hidden block, the representation the encoder *would* have
computed there. Not the pixels — the representation. That distinction is the whole point, and
section 6 comes back to it.

In [ ]:
from datasets import load_dataset

ds = load_dataset("blanchon/EuroSAT_RGB")["train"].shuffle(seed=0)
CLASSES = ds.features["label"].names
to_tensor = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True),
                        v2.Resize((IMG, IMG), antialias=True)])

t0 = time.time()
X = torch.stack([to_tensor(ds[i]["image"].convert("RGB")) for i in range(N_TRAIN)])
y = torch.tensor([ds[i]["label"] for i in range(N_TRAIN)])
print(f"{tuple(X.shape)} tiles in {time.time()-t0:.0f}s")


def sample_masks(rng, n_blocks=4, block=2):
    """Pick `n_blocks` non-overlapping block x block patch squares as targets.

    Returns (context_idx, target_idx). Both are fixed-length, which keeps every
    batch the same shape -- the simplification that makes this readable.
    """
    chosen, tries = set(), 0
    while len(chosen) < n_blocks * block * block and tries < 200:
        tries += 1
        r, c = rng.randint(0, GRID - block), rng.randint(0, GRID - block)
        blk = {(r + i) * GRID + (c + j) for i in range(block) for j in range(block)}
        if blk & chosen:
            continue
        chosen |= blk
    ctx = [i for i in range(GRID * GRID) if i not in chosen]
    return torch.tensor(ctx), torch.tensor(sorted(chosen))

In [ ]:
rng = random.Random(0)
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for ax, k in zip(axes, range(4)):
    _, tgt = sample_masks(rng)
    masked = X[k].clone()
    for p in tgt.tolist():
        r, c = divmod(p, GRID)
        masked[:, r * PATCH:(r + 1) * PATCH, c * PATCH:(c + 1) * PATCH] = 0.0
    ax.imshow(masked.permute(1, 2, 0))
    ax.set_title(f"{CLASSES[y[k]][:16]}\n{len(tgt)} patches hidden", fontsize=8)
    ax.axis("off")
plt.tight_layout(); plt.show()

The black squares are what the context encoder never sees. A second encoder — a slowly-trailing
copy — *does* see them, and its output at those positions is what we are trying to predict.

## 2. The two encoders and the predictor

In [ ]:
class Block(nn.Module):
    """Pre-norm transformer block: attention + feed-forward, both residual."""

    def __init__(self, dim: int, heads: int):
        super().__init__()
        self.n1, self.n2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(dim, dim * 4), nn.GELU(), nn.Linear(dim * 4, dim))

    def forward(self, x: Tensor) -> Tensor:
        h = self.n1(x)
        x = x + self.attn(h, h, h, need_weights=False)[0]
        return x + self.ff(self.n2(x))


class ViT(nn.Module):
    """Patch-embed -> add position -> transformer. Returns (B, n_patches, dim).

    `keep` selects which patches to encode at all -- the context encoder is handed
    only the visible ones, so it cannot cheat by looking at the targets.
    """

    def __init__(self, dim=DIM, depth=DEPTH, heads=HEADS):
        super().__init__()
        self.embed = nn.Conv2d(3, dim, PATCH, PATCH)
        self.pos = nn.Parameter(torch.zeros(1, GRID * GRID, dim))
        nn.init.trunc_normal_(self.pos, std=0.02)
        self.blocks = nn.ModuleList([Block(dim, heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)

    def forward(self, images: Tensor, keep: Tensor | None = None) -> Tensor:
        x = self.embed(images).flatten(2).transpose(1, 2) + self.pos
        if keep is not None:
            x = x[:, keep, :]
        for b in self.blocks:
            x = b(x)
        return self.norm(x)


class Predictor(nn.Module):
    """Context tokens + target positions -> predicted representations there.

    The `mask_token` is a single learned vector, reused for every prediction. What
    makes each prediction different is the **positional embedding added to it** --
    that is how the predictor is told *where* to look, and it is the piece a BYOL
    predictor does not have.
    """

    def __init__(self, dim: int = DIM, inner: int = 96, heads: int = 3, depth: int = 2):
        super().__init__()
        self.into = nn.Linear(dim, inner)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, inner))
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        self.pos = nn.Parameter(torch.zeros(1, GRID * GRID, inner))
        nn.init.trunc_normal_(self.pos, std=0.02)
        self.blocks = nn.ModuleList([Block(inner, heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(inner)
        self.out = nn.Linear(inner, dim)

    def forward(self, ctx: Tensor, ctx_idx: Tensor, tgt_idx: Tensor) -> Tensor:
        c = self.into(ctx) + self.pos[:, ctx_idx, :]
        q = self.mask_token.expand(ctx.shape[0], len(tgt_idx), -1) + self.pos[:, tgt_idx, :]
        x = torch.cat([c, q], dim=1)
        for b in self.blocks:
            x = b(x)
        return self.out(self.norm(x[:, -len(tgt_idx):, :]))

## 3. Training, with the instrument attached

`vectormesh.components.RepresentationStd` is the collapse detector from notebook 01. It is
logged from the first step this time — a loss curve alone cannot tell you whether the encoder is
learning or dying.

In [ ]:
from vectormesh.components import RepresentationStd

repstd = RepresentationStd(index=())     # our tokens are a plain tensor
HEALTHY = 1 / DIM**0.5
print(f"healthy representation std for dim={DIM}: {HEALTHY:.4f}")


def train_ijepa(steps, lr=5e-4, momentum=0.996, pred_dim=96, n_blocks=4,
                data=None, log_every=None, seed=0):
    """One I-JEPA run. Returns (encoder, history)."""
    data = X if data is None else data
    torch.manual_seed(seed)
    enc, pred = ViT(), Predictor(DIM, int(pred_dim))
    target_enc = copy.deepcopy(enc)
    for p in target_enc.parameters():
        p.requires_grad = False          # the target is averaged into, never trained

    opt = torch.optim.AdamW(list(enc.parameters()) + list(pred.parameters()), lr=lr)
    rng, hist = random.Random(seed), []

    for step in range(steps):
        imgs = data[torch.randint(0, len(data), (BATCH,))]
        ctx_idx, tgt_idx = sample_masks(rng, int(n_blocks))

        z_ctx = enc(imgs, keep=ctx_idx)                       # visible patches only
        with torch.no_grad():
            z_all = target_enc(imgs)                          # the trailing copy sees everything
            target = F.layer_norm(z_all[:, tgt_idx, :], (DIM,))

        loss = F.smooth_l1_loss(pred(z_ctx, ctx_idx, tgt_idx), target)
        opt.zero_grad(); loss.backward(); opt.step()

        # EMA, ramped toward 1: a target that moves early and freezes late
        m = momentum + (0.9999 - momentum) * (step / max(steps, 1))
        with torch.no_grad():
            for tp, op in zip(target_enc.parameters(), enc.parameters()):
                tp.mul_(m).add_(op.detach(), alpha=1 - m)

        every = log_every or max(steps // 12, 1)
        if step % every == 0 or step == steps - 1:      # always keep the last step
            hist.append((step, loss.item(), repstd(None, z_ctx)))
    return enc, hist

In [ ]:
t0 = time.time()
naive_enc, naive_hist = train_ijepa(STEPS)
print(f"trained in {time.time()-t0:.0f}s")

steps_, losses_, stds_ = zip(*naive_hist)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
a1.plot(steps_, losses_); a1.set_title("loss"); a1.set_xlabel("step")
a2.plot(steps_, stds_)
a2.axhline(HEALTHY, color="k", ls=":", lw=0.8, label=f"healthy ({HEALTHY:.3f})")
a2.set_title("representation std"); a2.set_xlabel("step"); a2.set_ylim(bottom=0); a2.legend()
plt.tight_layout(); plt.show()
print(f"final loss {losses_[-1]:.4f}   final repstd {stds_[-1]:.4f}   healthy {HEALTHY:.4f}")

## 4. The loss is delighted. The encoder is dead.

Read the two panels together, exactly as in notebook 01.

The loss falls towards zero, which looks like a model solving its task. The representation std
tells the real story: the encoder is producing nearly the same vector everywhere. And a constant
is *trivially predictable* — `F.layer_norm` maps a constant token to a fixed vector, so the
predictor learns that fixed vector and the loss goes to zero without anything having been learnt.

This is the same trap as notebook 01's missing stop-gradient, reached by a different route, and
it is worth stating as a rule:

> **A self-supervised loss going to zero is not evidence of success.** It is equally consistent
> with total failure. You need a second number.

## 5. So what do we search on?

The obvious move is to hyperparameter-search our way out — momentum, learning rate, predictor
width, how much we mask. Lesson 4 built exactly that machinery.

But scoring a search is now a genuine question, because **minimising the loss selects _for_
collapse.** The dead run above has a better loss than any healthy run will. Point `ray` at the
loss and it will find the deadest configuration in the space, efficiently and with confidence.

So we run the search twice, and compare what each one picks.

In [ ]:
from ray import tune
from ray.tune.schedulers import AsyncHyperBandScheduler
from ray.tune.search.hyperopt import HyperOptSearch


def quick_probe(encoder, data, labels, n=None):
    """Linear probe on frozen features. The honest signal -- and it needs labels.

    `n` is clamped to the data available: asking for more rows than exist yields an
    empty test split, a NaN score, and a search that reports no usable metric at all.
    """
    n = min(n or PROBE_N, len(data))
    encoder.eval()
    with torch.no_grad():
        E = torch.cat([encoder(data[i:i + 256]).mean(1) for i in range(0, n, 256)])
    E = (E - E.mean(0, keepdim=True)) / (E.std(0, keepdim=True) + 1e-6)
    ntr = n * 2 // 3
    clf = nn.Linear(E.shape[1], len(CLASSES))
    opt = torch.optim.Adam(clf.parameters(), lr=1e-2, weight_decay=1e-4)
    for _ in range(120):
        opt.zero_grad(); F.cross_entropy(clf(E[:ntr]), labels[:ntr]).backward(); opt.step()
    with torch.no_grad():
        return float((clf(E[ntr:n]).argmax(1) == labels[ntr:n]).float().mean())


SEARCH_SPACE = {
    "momentum": tune.loguniform(0.9, 0.9999),
    "lr": tune.loguniform(1e-4, 3e-3),
    "pred_dim": tune.choice([48, 96, 192]),
    "n_blocks": tune.choice([2, 4, 6]),
}

# A subset, so a trial is quick. Every trial reports all three numbers; only the
# `metric` below decides which one the search steers by.
Xs, ys = X[:2000], y[:2000]


def trial(config):
    enc, hist = train_ijepa(SEARCH_STEPS, lr=config["lr"], momentum=config["momentum"],
                            pred_dim=config["pred_dim"], n_blocks=config["n_blocks"],
                            data=Xs, log_every=10**9)
    with torch.no_grad():
        std = repstd(None, enc(Xs[:256]))
    tune.report({"loss": hist[-1][1],          # the TRAINED loss, not the initial one
                 "repstd": std,
                 "probe": quick_probe(enc, Xs, ys)})


def run_search(metric, mode):
    tuner = tune.Tuner(
        tune.with_resources(trial, {"cpu": 2}),
        param_space=SEARCH_SPACE,
        tune_config=tune.TuneConfig(
            # metric/mode live HERE and nowhere else: ray 2.55 raises if the
            # scheduler or the search algorithm also carry their own.
            metric=metric, mode=mode, num_samples=N_TRIALS,
            search_alg=HyperOptSearch(),
            scheduler=AsyncHyperBandScheduler(),
        ),
        run_config=tune.RunConfig(verbose=0),
    )
    return tuner.fit()


def show(results, metric, mode):
    df = results.get_dataframe().sort_values(metric, ascending=(mode == "min"))
    print(f"{'loss':>9s} {'repstd':>8s} {'probe':>7s} | {'momentum':>9s} {'lr':>9s} {'pdim':>5s} {'blk':>4s}")
    print("-" * 62)
    for _, r in df.head(6).iterrows():
        print(f"{r['loss']:9.4f} {r['repstd']:8.4f} {r['probe']:7.3f} | "
              f"{r['config/momentum']:9.4f} {r['config/lr']:9.5f} "
              f"{int(r['config/pred_dim']):5d} {int(r['config/n_blocks']):4d}")
    return results.get_best_result().config

In [ ]:
print("=== SEARCH A: scored on the training loss (the obvious choice) ===")
res_a = run_search("loss", "min")
cfg_a = show(res_a, "loss", "min")
print(f"\nwinner: {cfg_a}")

In [ ]:
print("=== SEARCH B: scored on a linear probe (needs labels) ===")
res_b = run_search("probe", "max")
cfg_b = show(res_b, "probe", "max")
print(f"\nwinner: {cfg_b}")

## 6. What each search chose

Compare the `repstd` column between the two tables — that is the whole section. One search is
steering toward configurations whose loss is small; the other toward configurations whose
features are useful. They are not the same configurations, and the `repstd` column shows which
of them are alive.

Two things follow, and the second one is bigger than this notebook:

1. **A search is only as good as its objective.** TPE is not doing anything wrong when it finds
   a collapsed encoder — it is finding exactly what it was asked for. The error is upstream, in
   the choice of what to score.
2. **Self-supervision removes labels from training, not from model selection.** The probe needs
   labels. You can pretrain on a million unlabelled tiles and still need a few hundred labelled
   ones to tell which pretraining run was any good. "Unsupervised" describes the gradient, not
   the project.

## 7. Train the winner properly, and test notebook 02's prediction

Now the payoff. Take the configuration search B selected, train it for longer, and ask the
question notebook 02 set up: does a **patch-level** objective produce better **per-patch**
features than an image-level one?

The comparison holds architecture fixed — same ViT, same data, same budget — and varies only
the objective.

In [ ]:
def info_nce(a, b, temp=0.2):
    a, b = F.normalize(a, dim=-1), F.normalize(b, dim=-1)
    return F.cross_entropy(a @ b.t() / temp, torch.arange(len(a)))


augment = v2.Compose([
    v2.RandomResizedCrop(IMG, scale=(0.4, 1.0), antialias=True),
    v2.RandomHorizontalFlip(),
    v2.RandomApply([v2.ColorJitter(brightness=0.4, contrast=0.4)], p=0.8),
])


def train_infonce(steps, lr=5e-4, seed=0):
    """The image-level control: same ViT, patches pooled away before the loss."""
    torch.manual_seed(seed)
    enc = ViT()
    opt = torch.optim.AdamW(enc.parameters(), lr=lr)
    hist = []
    for step in range(steps):
        imgs = X[torch.randint(0, len(X), (BATCH,))]
        z1, z2 = enc(augment(imgs)).mean(1), enc(augment(imgs)).mean(1)
        loss = (info_nce(z1, z2) + info_nce(z2, z1)) / 2
        opt.zero_grad(); loss.backward(); opt.step()
        if step % max(steps // 12, 1) == 0:
            hist.append((step, loss.item(), repstd(None, z1)))
    return enc, hist


t0 = time.time()
best_enc, best_hist = train_ijepa(FINAL_STEPS, **cfg_b)
print(f"I-JEPA  ({FINAL_STEPS} steps) loss {best_hist[-1][1]:.4f} "
      f"repstd {best_hist[-1][2]:.4f}  [{time.time()-t0:.0f}s]")

t0 = time.time()
nce_enc, nce_hist = train_infonce(FINAL_STEPS)
print(f"InfoNCE ({FINAL_STEPS} steps) loss {nce_hist[-1][1]:.4f} "
      f"repstd {nce_hist[-1][2]:.4f}  [{time.time()-t0:.0f}s]")

In [ ]:
# per-patch evaluation: 2x2 composites of quarter-size tiles. At 64px with 8px
# patches the quadrant boundary lands exactly on the patch-4 line, so no patch
# straddles two classes and there is nothing to exclude.
quarter = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True),
                      v2.Resize((IMG // 2, IMG // 2), antialias=True)])
pool_tiles = [(quarter(ds[i]["image"].convert("RGB")), ds[i]["label"])
              for i in range(N_TRAIN, N_TRAIN + param(3000, test=400))]

rng = random.Random(0)
comps, quads = [], []
for _ in range(param(1200, test=120)):
    p = [pool_tiles[rng.randint(0, len(pool_tiles) - 1)] for _ in range(4)]
    comps.append(torch.cat([torch.cat([p[0][0], p[1][0]], 2),
                            torch.cat([p[2][0], p[3][0]], 2)], 1))
    quads.append(torch.tensor([t[1] for t in p]))
Comp, Quad = torch.stack(comps), torch.stack(quads)

rows = torch.arange(GRID).view(GRID, 1).expand(GRID, GRID)
cols = torch.arange(GRID).view(1, GRID).expand(GRID, GRID)
quad_of_patch = ((rows >= GRID // 2).long() * 2 + (cols >= GRID // 2).long()).reshape(-1)
Ypatch = Quad[:, quad_of_patch]


def per_patch_probe(encoder):
    encoder.eval()
    with torch.no_grad():
        T = torch.cat([encoder(Comp[i:i + 128]) for i in range(0, len(Comp), 128)])
    ntr = len(T) * 2 // 3
    tr, te = T[:ntr].reshape(-1, DIM), T[ntr:].reshape(-1, DIM)
    ytr, yte = Ypatch[:ntr].reshape(-1), Ypatch[ntr:].reshape(-1)
    mu, sd = tr.mean(0, keepdim=True), tr.std(0, keepdim=True) + 1e-6
    clf = nn.Linear(DIM, len(CLASSES))
    opt = torch.optim.Adam(clf.parameters(), lr=1e-2, weight_decay=1e-4)
    for _ in range(200):
        opt.zero_grad(); F.cross_entropy(clf((tr - mu) / sd), ytr).backward(); opt.step()
    with torch.no_grad():
        return float((clf((te - mu) / sd).argmax(1) == yte).float().mean())


print(f"{'encoder':28s} {'per-patch':>10s} {'whole-tile':>11s} {'repstd':>8s}")
print("-" * 62)
for name, enc in [("random ViT (floor)", ViT()),
                  ("InfoNCE (image-level)", nce_enc),
                  ("I-JEPA (patch-level)", best_enc)]:
    with torch.no_grad():
        std = repstd(None, enc(X[:256]))
    print(f"{name:28s} {per_patch_probe(enc):10.3f} "
          f"{quick_probe(enc, X, y):11.3f} {std:8.4f}")

## What to read in that table

Three columns, three different questions:

- **per-patch** — the prediction under test. If notebook 02's rule holds, the patch-level
  objective should lead here.
- **whole-tile** — the task the *image-level* objective actually optimises. InfoNCE leading this
  column would not be a defeat; it would make the result a clean trade rather than a ranking.
- **repstd** — whether either encoder is alive at all. Check this before believing either
  of the other two columns.

Both encoders must clear the random-ViT floor for any of it to mean anything. A small ViT trained
for a few thousand steps on a laptop is a weak model — the question here is which *direction* the
objective pushes it, not whether it is any good in absolute terms.

## Exercises

**E1 — Break it on purpose.** Take the winning config and set `momentum=0.0`, so the target is
the online encoder itself with no trailing copy. Predict what happens to `repstd` before running it.

**E2 — Pixels instead of representations.** Change the target from `target_enc`'s output to the
raw *pixels* of the hidden patches. That is a masked autoencoder rather than a JEPA. Which one
gives better per-patch features here, and what does the loss curve look like in each case?

**E3 — The search that cannot fail.** Search A was scored on the loss and found collapse. Design
an objective that combines `repstd` and the probe into a single number, and argue for your
weighting. Then check whether it picks the same configuration search B did.

**E4 — How many labels does model selection need?** `quick_probe` uses 1200 labelled tiles. Try
600, 200, 60. At what point does the search stop being able to tell good configurations from bad
ones? This puts a number on the claim in section 6.

**E5 — Masking ratio.** `n_blocks` controls how much is hidden. Very little hidden makes the task
trivial; too much leaves no context. Plot per-patch probe accuracy against masked fraction and
find the useful range.